The Next Word Prediction models have a range of applications across various industries. For example, when you start typing a message on your phone, it suggests the next word to speed up your typing. Similarly, search engines predict and show search suggestions as you type in the search bar. Next word prediction helps us communicate faster and more accurately by anticipating what we might say or search for.

To build a Next Word Prediction model:

    1.start by collecting a diverse dataset of text documents, 
    2.preprocess the data by cleaning and tokenizing it, 
    3.prepare the data by creating input-output pairs, 
    4.engineer features such as word embeddings, 
    5.select an appropriate model like an LSTM or GPT, 
    6.train the model on the dataset while adjusting hyperparameters,
    7.improve the model by experimenting with different techniques and architectures.

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

file = r"Data\sherlock_holmes_stories_plain_text_advs.txt"

# Read the text file 
with open(file, 'r', encoding='utf-8') as file:
    text = file.read()

Now let's tokenize the text to create a sequence of words:

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])  
total_words = len(tokenizer.word_index) + 1

In the text is tokenized, which means it is divided into individual words or token. The 'Tokenizer' object is created, which will handle the tokenization process. The ‘fit_on_texts’ method of the tokenizer is called, passing the ‘text’ as input. This method analyzes the text and builds a vocabulary of unique words, assigning each word a numerical index. The ‘total_words’ variable is then assigned the value of the length of the word index plus one, representing the total number of distinct words in the text.

Now let’s create input-output pairs by splitting the text into sequences of tokens and forming n-grams from the sequences:

In [5]:
input_sequences = []

for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

the text data is split into lines using the ‘\n’ character as a delimiter. For each line in the text, the ‘texts_to_sequences’ method of the tokenizer is used to convert the line into a sequence of numerical tokens based on the previously created vocabulary. The resulting token list is then iterated over using a for loop. For each iteration, a subsequence, or n-gram, of tokens is extracted, ranging from the beginning of the token list up to the current index ‘i’

This n-gram sequence represents the input context, with the last token being the target or predicted word. This n-gram sequence is then appended to the ‘input_sequences’ list. This process is repeated for all lines in the text, generating multiple input-output sequences that will be used for training the next word prediction model.

Now let’s pad the input sequences to have equal length:

This loop reads one line at a time
Why line-by-line?
Language models learn better from sentence-like structures

In [ ]:
input_sequences

In [14]:
max_sequence_len = max([len(seq) for seq in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

 input sequences are padded to ensure all sequences have the same length. The variable ‘max_sequence_len’ is assigned the maximum length among all the input sequences. The ‘pad_sequences’ function is used to pad or truncate the input sequences to match this maximum length.

 The ‘pad_sequences’ function takes the input_sequences list, sets the maximum length to ‘max_sequence_len’, and specifies that the padding should be added at the beginning of each sequence using the ‘padding=pre’ argument. Finally, the input sequences are converted into a numpy array to facilitate further processing.


 ↓
 Text

 ↓
Tokenizer (words → numbers)

 ↓
N-grams (growing context)

 ↓
Padding (same length)

 ↓
X = context words
y = next word

 ↓
One-hot encoding (this line)

 ↓
Embedding (meaning vectors)

 ↓
LSTM (understanding context)

 ↓
Softmax (next word prediction)

 Now let’s split the sequences into input and output:

In [21]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# X = all words except the last one
# y = the last word (the word to predict)

the input sequences are split into two arrays, ‘X’ and ‘y’, to create the input and output for training the next word prediction model. The ‘X’ array is assigned the values of all rows in the ‘input_sequences’ array except for the last column. It means that ‘X’ contains all the tokens in each sequence except for the last one, representing the input context.

On the other hand, the ‘y’ array is assigned the values of the last column in the ‘input_sequences’ array, which represents the target or predicted word.

let’s convert the output to one-hot encode vectors:

In [16]:
y= np.array(tf.keras.utils.to_categorical(y, num_classes=total_words))

we are converting the output array into a suitable format for training a model, where each target word is represented as a binary vector.

let’s build a neural network architecture to train the model:

In [18]:
model = Sequential()
model.add(Embedding(total_words, 100, input_length=max_sequence_len-1))
model.add(LSTM(150))
model.add(Dense(total_words, activation='softmax'))
print(model.summary())

e:\1_Data_Science\venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


The code above defines the model architecture for the next word prediction model. The ‘Sequential’ model is created, which represents a linear stack of layers. The first layer added to the model is the ‘Embedding’ layer, which is responsible for converting the input sequences into dense vectors of fixed size. It takes three arguments:

‘total_words’, which represents the total number of distinct words in the vocabulary; 
‘100’, which denotes the dimensionality of the word embeddings; 
and ‘input_length’, which specifies the length of the input sequences.
The next layer added is the ‘LSTM’ layer, a type of recurrent neural network (RNN) layer designed for capturing sequential dependencies in the data. It has 150 units, which means it will learn 150 internal representations or memory cells.

Finally, the ‘Dense’ layer is added, which is a fully connected layer that produces the output predictions. It has ‘total_words’ units and uses the ‘softmax’ activation function to convert the predicted scores into probabilities, indicating the likelihood of each word being the next one in the sequence.

Now let’s compile and train the model:

In [19]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=100, verbose=1)

Epoch 1/100
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 102s 29ms/step - accuracy: 0.0766 - loss: 6.2498
Epoch 2/100
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 95s 31ms/step - accuracy: 0.1235 - loss: 5.5317
Epoch 3/100
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 98s 33ms/step - accuracy: 0.1468 - loss: 5.1504
Epoch 4/100
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 96s 32ms/step - accuracy: 0.1637 - loss: 4.8217
Epoch 5/100
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 94s 31ms/step - accuracy: 0.1813 - loss: 4.5158
Epoch 6/100
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 88s 29ms/step - accuracy: 0.1994 - loss: 4.2345
Epoch 7/100
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 92s 31ms/step - accuracy: 0.2244 - loss: 3.9641
Epoch 8/100
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 121s 40ms/step - accuracy: 0.2532 - loss: 3.7050
Epoch 9/100
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 90s 30ms/step - accuracy: 0.2862 - loss: 3.4604
Epoch 10/100
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 135s 27ms/step - accuracy: 0.3223 - loss: 3.2340
Epoch 11/100
3010/3010 ━━━━━━━━━━━━━━━━━━━━ 90s 30ms/step - accuracy: 0.3561 - loss: 3

In the above code, the model is being compiled and trained. The ‘compile’ method configures the model for training. The ‘loss’ parameter is set to ‘categorical_crossentropy’, a commonly used loss function for multi-class classification problems. The ‘optimizer’ parameter is set to ‘adam’, an optimization algorithm that adapts the learning rate during training.

The ‘metrics’ parameter is set to ‘accuracy’ to monitor the accuracy during training. After compiling the model, the ‘fit’ method is called to train the model on the input sequences ‘X’ and the corresponding output ‘y’. The ‘epochs’ parameter specifies the number of times the training process will iterate over the entire dataset. The ‘verbose’ parameter is set to ‘1’ to display the training process.

The above code will take more than an hour to execute. Once the code is executed, here’s how we can generate the next word predictions using our model:

In [22]:
seed_text = "I will leave if they"
next_words = 3 

for _ in range(next_words):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted = np.argmax(model.predict(token_list), axis=-1)
    output_word = ""
    for word, index in tokenizer.word_index.items():
        if index == predicted:
            output_word = word
            break
    seed_text += " " + output_word

print(seed_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 422ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
I will leave if they can trust the


The above code generates the next word predictions based on a given seed text. The ‘seed_text’ variable holds the initial text. The ‘next_words’ variable determines the number of predictions to be generated. Inside the for loop, the ‘seed_text’ is converted into a sequence of tokens using the tokenizer. The token sequence is padded to match the maximum sequence length.

The model predicts the next word by calling the ‘predict’ method on the model with the padded token sequence. The predicted word is obtained by finding the word with the highest probability score using ‘np.argmax’. Then, the predicted word is appended to the ‘seed_text’, and the process is repeated for the desired number of ‘next_words’. Finally, the ‘seed_text’ is printed, which contains the initial text followed by the generated predictions.

In [25]:
model.save("Next_word_prediction_model.h5")
print("This saves Model architecture, Learned weights, Optimizer state")
model.save("Next_word_prediction_model.keras")


This saves Model architecture, Learned weights, Optimizer state


the above code save the model code with architecture, learned weights, optimizer state and pickle code saves the tokenizer output parameters

json file saves the code params this are the required files to run your model without repeat training train once and run as much time you want. 

In [26]:
# Save the Tokenizer
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Tokenizer saved as tokenizer.pkl")

Tokenizer saved as tokenizer.pkl


In [27]:
# Save Parameters
import json

config = {
    "max_sequence_len": max_sequence_len,
    "total_words": total_words
}

with open("config.json", "w") as f:
    json.dump(config, f)

print("Config saved")

Config saved


To avoid confusions keep the files in a separate folder and run it.